# Embedding Drift (Pipeline B)

**Forensic question:** Does the semantic fingerprint of each Mediaite author drift toward AI-like text over time?

**Pipeline B status:** With Phase 15 fixes E/F/G and the regenerated `data/embeddings/manifest.jsonl`, Pipeline B is now alive for 12/12 tracked named authors. Newsroom-wide there are 8,042 drift-only persisted convergence windows; Per-author `pb_max` (named authors) ranges from 0.520 (zachary-leeman) to 0.598 (david-gilmour). `tommy-christopher` leads in raw drift-only window count (1,070), and `michael-luciano` is the largest drift-only signal with no AB confirmation (958 drift-only / 0 AB). `colby-hall` is the one author with substantial AB-confirmed windows (170), making colby-hall's signal the most cross-validated of the cohort.

**Input artifacts:**
- `data/analysis/<slug>_drift.json` — `monthly_centroid_velocities`, `intra_period_variance_trend`, `baseline_centroid_similarity`, `ai_baseline_similarity`
- `data/analysis/<slug>_baseline_curve.json` — author-specific baseline trajectory
- `data/analysis/<slug>_centroids.npz` — centroid vectors per period
- `data/analysis/<slug>_result.json::convergence_windows[].pipeline_b_score` and `passes_via` (now includes `drift_only`)

**Output artifacts:**
- Per-author summary table (pb_max / drift_only / ab / ratio / total)
- Monthly centroid velocity plot for the top-3 authors by drift-only count
- Distribution of `pipeline_b_score` across all persisted windows, faceted by author
- Diagnostic report on missing drift artifacts (E2 from PR #67)

**Run metadata:** (auto-populated by first code cell)


In [ ]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+notebook_connected+png'


In [ ]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+notebook_connected+png'


In [ ]:
# Parameters — override via: quarto render NOTEBOOK -P author_slug:some-slug
author_slug = "all"

In [ ]:
# | echo: false
import sys
from datetime import UTC, datetime

from IPython.display import Markdown, display

from forensics.config import get_settings
from forensics.utils.provenance import compute_config_hash, compute_corpus_hash

_settings = get_settings()
config_hash = compute_config_hash(_settings)
corpus_hash = compute_corpus_hash(_settings.db_path)
run_timestamp = datetime.now(UTC).isoformat()
display(
    Markdown(f"""
| Key | Value |
|-----|-------|
| Config hash | `{config_hash}` |
| Corpus hash | `{corpus_hash}` |
| Run timestamp | `{run_timestamp}` |
| Python | `{sys.version}` |
""")
)

## Methodology — Phase 15 fixes that revived Pipeline B

Pipeline B (embedding-drift convergence) used to fire for only 1/14 authors. Three fixes in Phase 15 brought it to 14/14:

- **Fix-E (percentile mode for AI-baseline similarity).** Author-vs-AI centroid similarity is now scored against a percentile of the per-author similarity distribution rather than a hard absolute cutoff. This stops the test from collapsing on authors whose absolute similarity to the AI baseline sits below 0.5 but whose distribution still has a clear right-tail.
- **Fix-F (threshold drop, 0.5 → 0.3).** The `pipeline_b_score` threshold for declaring a window "drifting" was lowered from 0.5 to 0.3, calibrated against the empirical distribution. With the percentile-mode change above, this threshold now distinguishes signal from noise without admitting every window.
- **Fix-G (drift-only channel).** A window can now persist as Pipeline-B-positive via the `drift_only` channel in `passes_via` even when Pipeline A's lexical/structural ratio test does not also pass. This separates the embedding signal from the lexical signal so we can study them independently. AB-confirmed windows (where both fire) remain the strongest evidence.

The fourth load-bearing input — `data/embeddings/manifest.jsonl` — was regenerated to cover all 14 authors. Without this manifest, `load_drift_summary` silently returned empty fields for any author whose embeddings were not registered, even when their `.npy` files existed on disk. Phase 15 E2 (PR #67) adds a WARNING in `load_drift_summary` whenever a cached drift artifact is missing but the author has embeddings on disk, so this class of silent failure now surfaces in logs.


In [ ]:
import json

import numpy as np
import polars as pl
import plotly.graph_objects as go

from forensics.config import get_project_root, get_settings
from forensics.utils.charts import register_forensics_template

register_forensics_template()

settings = get_settings()
ROOT = get_project_root()
ANALYSIS_DIR = ROOT / "data" / "analysis"
EMBEDDINGS_DIR = ROOT / "data" / "embeddings"


def _slug_set() -> list[str]:
    """Return author slugs that have a result.json on disk, in deterministic order."""
    SHARED_BYLINE_SLUGS = {"mediaite", "mediaite-staff"}
    return sorted(
        {
            p.name.removesuffix("_result.json")
            for p in ANALYSIS_DIR.glob("*_result.json")
            if p.name.removesuffix("_result.json") not in SHARED_BYLINE_SLUGS
        }
    )


def _load_result(slug: str) -> dict:
    return json.loads((ANALYSIS_DIR / f"{slug}_result.json").read_text(encoding="utf-8"))


def _load_drift(slug: str) -> dict:
    p = ANALYSIS_DIR / f"{slug}_drift.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.is_file() else {}


def _load_centroid_months(slug: str) -> list[str]:
    p = ANALYSIS_DIR / f"{slug}_centroids.npz"
    if not p.is_file():
        return []
    with np.load(p) as npz:
        return [str(m) for m in npz["months"].tolist()]


SLUGS = _slug_set()
print(f"Authors with result.json on disk: {len(SLUGS)}")
print(SLUGS)

# Honor the parameter cell: when author_slug != "all", restrict subsequent analysis
# but always compute the newsroom-wide table for context.
if author_slug != "all" and author_slug not in SLUGS:
    print(f"\nWARNING: author_slug={author_slug!r} is not in the on-disk slug set; "
          "falling back to 'all'.")
    author_slug = "all"
print(f"\nactive author_slug: {author_slug}")

## Per-author Pipeline B summary

For each author we surface:

- `pb_max` — maximum `pipeline_b_score` across persisted convergence windows
- `drift_only_count` — windows that persist via the new `drift_only` channel only
- `ab_count` — windows that pass via `ab` (lexical ratio AND embedding drift) — strongest evidence
- `ratio_count` — windows that pass via `ratio` (lexical/family ratio test)
- `total_windows` — total persisted convergence windows for the author

Sorted by `drift_only_count` descending so the heaviest drift signal appears at the top.


In [ ]:
def _build_summary_table(slugs: list[str]) -> pl.DataFrame:
    rows: list[dict] = []
    for slug in slugs:
        result = _load_result(slug)
        windows = result.get("convergence_windows", []) or []
        pb_scores = [
            w["pipeline_b_score"]
            for w in windows
            if w.get("pipeline_b_score") is not None
        ]
        pb_max = max(pb_scores) if pb_scores else 0.0
        drift_only = sum(1 for w in windows if "drift_only" in (w.get("passes_via") or []))
        ab = sum(1 for w in windows if "ab" in (w.get("passes_via") or []))
        ratio = sum(1 for w in windows if "ratio" in (w.get("passes_via") or []))
        rows.append(
            {
                "author_slug": slug,
                "pb_max": round(float(pb_max), 4),
                "drift_only_count": int(drift_only),
                "ab_count": int(ab),
                "ratio_count": int(ratio),
                "total_windows": int(len(windows)),
            }
        )
    return pl.DataFrame(rows).sort("drift_only_count", descending=True)


SUMMARY = _build_summary_table(SLUGS)

newsroom_drift_only = int(SUMMARY["drift_only_count"].sum())
authors_pb_positive = int((SUMMARY["pb_max"] > 0).sum())
print(
    f"Newsroom-wide drift-only persisted windows: {newsroom_drift_only:,}\n"
    f"Authors with pb_max > 0: {authors_pb_positive}/{len(SUMMARY)}\n"
    f"pb_max range: {SUMMARY['pb_max'].min():.3f} ({SUMMARY.sort('pb_max').row(0, named=True)['author_slug']}) "
    f"to {SUMMARY['pb_max'].max():.3f} ({SUMMARY.sort('pb_max', descending=True).row(0, named=True)['author_slug']})"
)

with pl.Config(tbl_rows=20, tbl_cols=10, fmt_str_lengths=40):
    print()
    print(SUMMARY)

## Monthly centroid velocity — top-3 authors by drift-only count

For the three authors with the highest drift-only window counts, plot the month-over-month
centroid velocity (cosine distance between consecutive monthly centroids). Velocity spikes
mark months in which the author's average semantic fingerprint moved sharply.

When `author_slug` is overridden via `-P`, the chart shows that author only.


In [ ]:
def _velocity_frame(slug: str) -> pl.DataFrame:
    """Pair monthly centroid velocities with their target month label.

    `monthly_centroid_velocities[i]` is the cosine distance between centroids of
    `months[i]` and `months[i+1]`; we plot it against `months[i+1]`.
    """
    drift = _load_drift(slug)
    velocities = drift.get("monthly_centroid_velocities") or []
    months = _load_centroid_months(slug)
    if len(months) >= 2 and len(velocities) >= 1:
        labels = months[1 : 1 + len(velocities)]
    else:
        labels = [f"m{i + 1}" for i in range(len(velocities))]
    if not velocities:
        return pl.DataFrame(
            {"author_slug": [], "month": [], "velocity": []},
            schema={"author_slug": pl.Utf8, "month": pl.Utf8, "velocity": pl.Float64},
        )
    return pl.DataFrame(
        {
            "author_slug": [slug] * len(velocities),
            "month": labels,
            "velocity": velocities,
        }
    )


if author_slug == "all":
    plot_slugs = SUMMARY.head(3)["author_slug"].to_list()
else:
    plot_slugs = [author_slug]

print(f"Plotting monthly centroid velocity for: {plot_slugs}")

velocity_frames = [_velocity_frame(s) for s in plot_slugs]
velocity_long = pl.concat(
    [f for f in velocity_frames if f.height > 0], how="vertical_relaxed"
)

fig_velocity = go.Figure()
for slug in plot_slugs:
    sub = velocity_long.filter(pl.col("author_slug") == slug).sort("month")
    if sub.height == 0:
        continue
    fig_velocity.add_trace(
        go.Scatter(
            x=sub["month"].to_list(),
            y=sub["velocity"].to_list(),
            mode="lines+markers",
            name=slug,
            hovertemplate="<b>%{fullData.name}</b><br>month=%{x}<br>velocity=%{y:.4f}<extra></extra>",
        )
    )

fig_velocity.update_layout(
    title="Monthly centroid velocity (cosine distance between consecutive monthly centroids)",
    xaxis_title="month",
    yaxis_title="centroid velocity",
    legend_title="author_slug",
    height=480,
)
fig_velocity.show()

## Distribution of `pipeline_b_score` across persisted windows

Histogram of `pipeline_b_score` for every persisted convergence window across all 14 authors,
faceted (overlaid) by author. The Fix-F threshold of 0.3 is drawn as a vertical reference; any
window to the right of it would qualify as drift-positive. The visible right-tail is what
Fix-G's `drift_only` channel now lets us harvest as Pipeline-B signal.


In [ ]:
FIX_F_THRESHOLD = 0.3


def _all_pb_scores(slugs: list[str]) -> pl.DataFrame:
    rows: list[dict] = []
    for slug in slugs:
        result = _load_result(slug)
        for w in result.get("convergence_windows", []) or []:
            score = w.get("pipeline_b_score")
            if score is None:
                continue
            rows.append({"author_slug": slug, "pipeline_b_score": float(score)})
    return pl.DataFrame(
        rows,
        schema={"author_slug": pl.Utf8, "pipeline_b_score": pl.Float64},
    )


hist_slugs = SLUGS if author_slug == "all" else [author_slug]
PB_SCORES = _all_pb_scores(hist_slugs)

print(
    f"Persisted windows with non-null pipeline_b_score: {PB_SCORES.height:,} "
    f"across {PB_SCORES['author_slug'].n_unique()} authors\n"
    f"Median pb: {PB_SCORES['pipeline_b_score'].median():.3f}; "
    f"P90: {PB_SCORES['pipeline_b_score'].quantile(0.9):.3f}; "
    f"max: {PB_SCORES['pipeline_b_score'].max():.3f}\n"
    f"Share above Fix-F threshold ({FIX_F_THRESHOLD}): "
    f"{(PB_SCORES['pipeline_b_score'] >= FIX_F_THRESHOLD).mean():.1%}"
)

fig_hist = go.Figure()
# Stable author ordering: by drift_only_count desc (matches summary table).
order = SUMMARY["author_slug"].to_list()
for slug in order:
    sub = PB_SCORES.filter(pl.col("author_slug") == slug)
    if sub.height == 0:
        continue
    fig_hist.add_trace(
        go.Histogram(
            x=sub["pipeline_b_score"].to_list(),
            name=slug,
            opacity=0.55,
            xbins={"start": 0.0, "end": 1.0, "size": 0.025},
            hovertemplate="<b>%{fullData.name}</b><br>pb=%{x}<br>count=%{y}<extra></extra>",
        )
    )

fig_hist.add_vline(
    x=FIX_F_THRESHOLD,
    line_dash="dash",
    line_color="#444",
    annotation_text=f"Fix-F threshold = {FIX_F_THRESHOLD}",
    annotation_position="top right",
)
fig_hist.update_layout(
    title="Distribution of pipeline_b_score across persisted windows (faceted by author)",
    barmode="overlay",
    xaxis_title="pipeline_b_score",
    yaxis_title="count",
    legend_title="author_slug",
    height=520,
)
fig_hist.show()

## Diagnostic block — drift artifact presence

Phase 15 E2 (PR #67) added a stable WARNING template in `forensics.analysis.drift.load_drift_summary`:

```
drift summary: missing artifact <label> for slug=<slug> but embeddings exist on disk
```

This cell re-checks artifact presence directly (without re-running `load_drift_summary`), so we can surface
silent drift-artifact write failures even when the analysis stage itself ran without errors. A clean run
should report `0` missing artifacts.


In [ ]:
def _author_has_embeddings_on_disk(slug: str) -> bool:
    slug_dir = EMBEDDINGS_DIR / slug
    if not slug_dir.is_dir():
        return False
    return any(slug_dir.iterdir())


def _check_drift_artifacts(slugs: list[str]) -> pl.DataFrame:
    rows: list[dict] = []
    artifact_specs = [
        ("drift.json", lambda s: ANALYSIS_DIR / f"{s}_drift.json"),
        ("baseline_curve.json", lambda s: ANALYSIS_DIR / f"{s}_baseline_curve.json"),
        ("centroids.npz", lambda s: ANALYSIS_DIR / f"{s}_centroids.npz"),
    ]
    for slug in slugs:
        has_emb = _author_has_embeddings_on_disk(slug)
        for label, path_fn in artifact_specs:
            p = path_fn(slug)
            present = p.is_file()
            would_warn = (not present) and has_emb
            rows.append(
                {
                    "author_slug": slug,
                    "artifact": label,
                    "present": present,
                    "embeddings_on_disk": has_emb,
                    "would_warn": would_warn,
                }
            )
    return pl.DataFrame(rows)


diag_slugs = SLUGS if author_slug == "all" else [author_slug]
DIAG = _check_drift_artifacts(diag_slugs)

missing = DIAG.filter(~pl.col("present"))
warnings_now = DIAG.filter(pl.col("would_warn"))

print(f"Authors checked: {DIAG['author_slug'].n_unique()}")
print(f"Total artifact slots: {DIAG.height} (3 per author)")
print(f"Missing artifacts: {missing.height}")
print(f"Would emit drift-artifact-missing WARNING (E2 trigger): {warnings_now.height}")

if warnings_now.height:
    print("\nWARNINGs that load_drift_summary would emit on next call:")
    with pl.Config(tbl_rows=50, tbl_cols=10, fmt_str_lengths=60):
        print(
            warnings_now.select(["author_slug", "artifact"]).sort(
                ["author_slug", "artifact"]
            )
        )
else:
    print("\nClean run: every author with embeddings on disk has all three drift artifacts.")

## Summary finding

After Phase 15 fixes E (percentile mode for AI-baseline similarity), F (threshold drop 0.5 → 0.3), and G (drift-only channel) — combined with the regenerated `data/embeddings/manifest.jsonl` covering all 14 authors — Pipeline B (embedding drift) now persists windows for **12/12** named Mediaite authors instead of **1/14**.

- Newsroom-wide drift-only persisted windows: **8,042**.
- `pb_max` (named authors) ranges from **0.520** (zachary-leeman) to **0.598** (david-gilmour); every named author clears `pb_max > 0`.
- `tommy-christopher` has the highest raw drift-only count (**1,070**); `michael-luciano` is the largest drift-only-without-AB signal (**958** drift-only / **0** AB).
- `colby-hall` is the only author with substantial AB-confirmed windows (**170**), making colby-hall's signal the most cross-validated of the cohort.

Read against notebook 04 (lexical features) and 07 (statistical evidence): the embedding-drift signal complements the lexical/structural ratio test and is now visible as an independent channel for every author in the cohort.
